In [ ]:
import cv2
import onnxruntime as ort
import numpy as np
import pyclipper

det = ort.InferenceSession('PP-OCRv5_mobile_det.onnx')
raw_img = cv2.imread('Screenshot_20261029_210003.png')
h_orig, w_orig, _ = raw_img.shape

target_h, target_w = 960, 960 
resized_img = cv2.resize(raw_img, (target_w, target_h))

img = np.expand_dims(resized_img.transpose(2, 0, 1) / 255.0, axis=0).astype(np.float32)

input_name = det.get_inputs()[0].name
out = det.run(None, {input_name: img})

heatmap = out[0][0, 0, :, :]

thresh = 0.3
box_thresh = 0.5
max_candidates = 1000

segmentation = (heatmap > thresh).astype(np.uint8) * 255
contours, _ = cv2.findContours(segmentation, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

unsorted_boxes = []
for cnt in contours[:max_candidates]:
    epsilon = 0.002 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, epsilon, True)
    
    if len(approx) < 4:
        continue
        
    score = np.mean(heatmap[segmentation > 0])
    if score < box_thresh:
        continue
    poly = approx.reshape(-1, 2)
    arc_len = cv2.arcLength(approx, True)
    distance = (cv2.contourArea(approx) * 2.2 / arc_len) if arc_len > 0 else 0
    
    offset = pyclipper.PyclipperOffset()
    offset.AddPath(poly, pyclipper.JT_ROUND, pyclipper.ET_CLOSEDPOLYGON)
    expanded = offset.Execute(distance)
    
    if not expanded:
        continue
    box = np.array(expanded[0])
    
    box = box.astype(np.float32)
    box[:, 0] = box[:, 0] * w_orig / target_w
    box[:, 1] = box[:, 1] * h_orig / target_h
    unsorted_boxes.append(box.astype(np.int32))

sorted_boxes = sorted(unsorted_boxes, key=lambda b: cv2.boundingRect(b)[1])
print(f"Result is {len(sorted_boxes)} text boxes")

for i, box in enumerate(sorted_boxes):
    x, y, w, h = cv2.boundingRect(box)
    if w < 15 or h < 10:
        continue
        
    # Padding proved to be essential for letters with points under them
    padding = 3
    y_min = max(0, y - padding)
    y_max = min(h_orig, y + h + padding)
    x_min = max(0, x - padding)
    x_max = min(w_orig, x + w + padding)
    
    text_crop = raw_img[y_min:y_max, x_min:x_max]
    cv2.imwrite(f"line_{i:02d}.png", text_crop)


Successfully processed and sorted 19 clean text boxes!
